In [51]:
Fabric_silver_path='abfss://b0cceb94-d226-4dbd-a75b-789c5defa0a2@onelake.dfs.fabric.microsoft.com/f5ac5edb-ff4b-4a7b-ab7c-1b4182e0d30b/Tables/tblsales_silver'
from pyspark.sql.functions import col, sum as _sum, avg, year, when, to_date, regexp_extract
from pyspark.sql import functions as F
silver_df=spark.read.format('delta').load(Fabric_silver_path)

StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 53, Finished, Available, Finished)

In [16]:
# STEP 1: Inspect schema and confirm data types
silver_df.printSchema()

StatementMeta(, fbeab874-064c-4ce7-8c78-228dd5b4f42c, 18, Finished, Available, Finished)

root
 |-- REF_DATE: date (nullable = true)
 |-- GEO: string (nullable = true)
 |-- DGUID: string (nullable = true)
 |-- Violations: string (nullable = true)
 |-- Actual_incidents: integer (nullable = true)
 |-- Cleared_by_charge: integer (nullable = true)
 |-- Cleared_otherwise: integer (nullable = true)
 |-- Percent_unfounded: double (nullable = true)
 |-- Percentage_change_in_rate: double (nullable = true)
 |-- Rate_per_100000_population: double (nullable = true)
 |-- Rate_adult_charged_per_100000_population_18_plus: double (nullable = true)
 |-- Rate_total_persons_charged_per_100000_population_12_plus: double (nullable = true)
 |-- Rate_youth_charged_per_100000_population_12_to_17: double (nullable = true)
 |-- Rate_youth_not_charged_per_100000_population_12_to_17: double (nullable = true)
 |-- Total_cleared: integer (nullable = true)
 |-- Total_adult_charged: integer (nullable = true)
 |-- Total_persons_charged: integer (nullable = true)
 |-- Total_youth_charged: integer (nullable 

In [41]:
# STEP 2: Cast numeric columns to double (only if they are strings)
numeric_cols = [
    "Actual_incidents", "Cleared_by_charge", "Cleared_otherwise", "Total_cleared",
    "Percent_unfounded", "Unfounded_incidents", "Total_adult_charged",
    "Total_youth_charged", "Total_persons_charged", "Rate_per_100000_population",
    "Rate_adult_charged_per_100000_population_18_plus",
    "Rate_youth_charged_per_100000_population_12_to_17",
    "Rate_youth_not_charged_per_100000_population_12_to_17",
    "Percentage_change_in_rate"
]

for col_name in numeric_cols:
    silver_df = silver_df.withColumn(col_name, col(col_name).cast("double"))

StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 43, Finished, Available, Finished)

In [81]:
# STEP 3: Create Dim_Year, Dim_Municipality, Dim_CrimeType

coords_df = spark.read.csv(
    "abfss://b0cceb94-d226-4dbd-a75b-789c5defa0a2@onelake.dfs.fabric.microsoft.com/f75ade0f-6d7e-4112-b6d9-91609ce7433d/Files/ontario_municipalities_with_coords.csv",
    header=True,
    inferSchema=True
)

# Dim_Year
Dim_Year = (
    silver_df
    .select(year(col("REF_DATE")).alias("YearKey"))
    .distinct()
    .orderBy("YearKey")
)

# Dim_Municipality
Dim_Municipality = silver_df.withColumn(
    "Area_Type",
    F.when(
        F.trim(F.regexp_extract(F.col("GEO"), r",\s*([^,\[]+)\s*\[\d+\]$", 1)) == "Ontario",
        # extract everything before the comma
        F.regexp_extract(F.col("GEO"), r"^([^,]+)", 1)
    ).otherwise(
        # original extracted area type
        F.trim(F.regexp_extract(F.col("GEO"), r",\s*([^,\[]+)\s*\[\d+\]$", 1))
    )
).select(
    col("DGUID").alias("MunicipalityKey"),
    col("GEO").alias("MunicipalityName"),
    col("Area_Type").alias("AreaType")
).join(coords_df, on="MunicipalityKey" , how="left").drop(coords_df['MunicipalityName']).select(
    'MunicipalityKey',
    'MunicipalityName',
    'AreaType',
    'CleanName',
    'Latitude',
    'Longitude'
).distinct()

display(Dim_Municipality)


StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 83, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, b40e579d-8ebc-4085-91e0-bb041adb71b7)

## Creating the Violations Table

In [61]:
abfs_path='abfss://b0cceb94-d226-4dbd-a75b-789c5defa0a2@onelake.dfs.fabric.microsoft.com/f5ac5edb-ff4b-4a7b-ab7c-1b4182e0d30b/Files/Raw/crime_data_metadata.csv'


StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 63, Finished, Available, Finished)

In [82]:
from pyspark.sql.types import StructType, StructField, StringType

violations_schema = StructType([
    StructField("Dimension_ID", StringType(), True),
    StructField("Member_Name", StringType(), True),
    StructField("Classification_Code", StringType(), True),
    StructField("Member_ID", StringType(), True),
    StructField("Parent_Name", StringType(), True),
    StructField("Parent_Member_ID", StringType(), True)
])

StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 84, Finished, Available, Finished)

In [83]:
df_violations=spark.read.format('csv').option('header','True').schema(violations_schema).load(abfs_path)

StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 85, Finished, Available, Finished)

In [64]:
display(df_violations)

StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 66, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 12d70220-0d7a-4ac2-a68a-41c64f9911f3)

In [65]:
df_violations.createOrReplaceTempView('t_violations_data')

StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 67, Finished, Available, Finished)

In [66]:
%%sql
SELECT * from t_violations_data

StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 68, Finished, Available, Finished)

<Spark SQL result set with 314 rows and 6 fields>

In [86]:
Fabric_tbl_violations="abfss://b0cceb94-d226-4dbd-a75b-789c5defa0a2@onelake.dfs.fabric.microsoft.com/f5ac5edb-ff4b-4a7b-ab7c-1b4182e0d30b/Tables/dim_violations"
try:
    spark.read.format('delta').load(Fabric_tbl_violations).createOrReplaceTempView('t_dim_violations_data')
except:
    v_create_violations_table=f"""
    CREATE TABLE IF NOT EXISTS dim_violations (
    Dimension_ID STRING,
    Member_Name STRING,
    Classification_Code STRING,
    Member_ID STRING,
    Parent_Name STRING,
    Parent_Member_ID STRING
    )
    USING DELTA
    """
    spark.sql(v_create_violations_table)
    spark.read.format('delta').load(Fabric_tbl_violations).createOrReplaceTempView('t_dim_violations_data')

StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 88, Finished, Available, Finished)

In [87]:
sql_statement_2 = f"""MERGE INTO dim_violations AS target
USING t_violations_data AS source
    ON target.Member_ID = source.Member_ID

WHEN MATCHED THEN 
    UPDATE SET 
        target.Dimension_ID = source.Dimension_ID,
        target.Member_Name = source.Member_Name,
        target.Classification_Code = source.Classification_Code,
        target.Parent_Name = source.Parent_Name,
        target.Parent_Member_ID = source.Parent_Member_ID
WHEN NOT MATCHED THEN 
    INSERT (Dimension_ID, Member_Name, Classification_Code, Member_ID, Parent_Name, Parent_Member_ID)
    VALUES (source.Dimension_ID, source.Member_Name, source.Classification_Code, source.Member_ID, source.Parent_Name, source.Parent_Member_ID);"""

spark.sql(sql_statement_2).show()

StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 89, Finished, Available, Finished)

AnalysisException: [DELTA_MERGE_UNRESOLVED_EXPRESSION] Cannot resolve target.Parent_Member_ID in UPDATE clause given columns target.Dimension_ID, target.Member_Name, target.Classification_Code, target.Member_ID, target.Parent_Name; line 1 pos 0

In [69]:
spark.sql("""
ALTER TABLE dim_violations SET TBLPROPERTIES (
   'delta.columnMapping.mode' = 'name',
   'delta.minReaderVersion' = '2',
   'delta.minWriterVersion' = '5'
)
""")
spark.sql("""
ALTER TABLE dim_violations
DROP COLUMN Parent_Name
""")

spark.sql("""
ALTER TABLE dim_violations
RENAME COLUMN Parent_Member_ID TO Parent_Name
""")


StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 71, Finished, Available, Finished)

DataFrame[]

In [70]:
spark.sql("""
UPDATE dim_violations
SET Parent_Name = 'Total prostitution'
WHERE Member_Name = 'Procuring'
""")

StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 72, Finished, Available, Finished)

DataFrame[num_affected_rows: bigint]

In [71]:
spark.sql("""
DELETE FROM dim_violations
WHERE Member_Name = 'Procuring'
AND Member_ID = '241'
""")

StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 73, Finished, Available, Finished)

DataFrame[num_affected_rows: bigint]

In [76]:
# STEP 4: Aggregate data for Fact_CrimeStats
Fact_CrimeStats = silver_df.withColumn(
    "ViolationCode",
    regexp_extract(col("Violations"), r"(\[\d+\])$", 1)
).groupBy(
    year(col("REF_DATE")).alias("YearKey"),
    col("DGUID").alias("MunicipalityKey"),
    col("ViolationCode").alias("CrimeTypeKey"),
).agg(
    _sum(col("Actual_incidents")).alias("ActualIncidents"),
    _sum(col("Cleared_by_charge")).alias("ClearanceByCharge"),
    _sum(col("Cleared_otherwise")).alias("ClearanceOtherwise"),
    _sum(col("Total_cleared")).alias("TotalCleared"),
    avg(col("Percent_unfounded")).alias("PercentUnfounded"),
    _sum(col("Unfounded_incidents")).alias("UnfoundedIncidents"),
    _sum(col("Total_adult_charged")).alias("AdultCharges"),
    _sum(col("Total_youth_charged")).alias("YouthCharges"),
    _sum(col("Total_persons_charged")).alias("PersonsCharged"),
    avg(col("Rate_per_100000_population")).alias("CrimeRatePer100k"),
    avg(col("Rate_adult_charged_per_100000_population_18_plus")).alias("RateAdultCharged"),
    avg(col("Rate_youth_charged_per_100000_population_12_to_17")).alias("RateYouthCharged"),
    avg(col("Rate_youth_not_charged_per_100000_population_12_to_17")).alias("RateYouthNotCharged"),
    avg(col("Percentage_change_in_rate")).alias("PercentageChangeRate")
).withColumn(
    "ClearanceRate",
    when(col("ActualIncidents") > 0, col("TotalCleared") / col("ActualIncidents")).otherwise(None)
)



StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 78, Finished, Available, Finished)

In [74]:
display(Fact_CrimeStats)

StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 76, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 07de417b-1391-47d3-8e8f-067ea17d69bb)

In [84]:
# STEP 5: Save Gold layer tables
Dim_Year.write.format("delta").mode("overwrite").saveAsTable("Dim_Year")
Dim_Municipality.write.format("delta").mode("overwrite").saveAsTable("Dim_Municipality")
Fact_CrimeStats.write.format("delta").mode("overwrite").saveAsTable("Fact_CrimeStats")

StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 86, Finished, Available, Finished)

In [85]:
# STEP 6: Display tables in Fabric-style interactive view

# Dim_Year
display(spark.table("Dim_Year"))

# Dim_Municipality
display(spark.table("Dim_Municipality"))

# Dim_Violations
display(spark.table("Dim_Violations"))

# Fact_CrimeStats
display(spark.table("Fact_CrimeStats"))


StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 87, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, f2fa3423-7946-4c73-83f7-052253c7ef36)

SynapseWidget(Synapse.DataFrame, 55cf2a7d-e5ee-4f1c-b30a-c5d5c24e85ab)

SynapseWidget(Synapse.DataFrame, 3e6456dc-41e8-4f85-9ee4-070f55a3f200)

SynapseWidget(Synapse.DataFrame, 437cd388-cb3c-4771-aa60-5d8cccc03ec8)

In [89]:
from pyspark.sql.functions import col, sum as _sum, avg

# Load Gold Layer tables
Fact_CrimeStats   = spark.read.table("Fact_CrimeStats")
Dim_Municipality  = spark.read.table("Dim_Municipality")
Dim_Violations     = spark.read.table("Dim_Violations")

# Join Fact with Municipality and CrimeType dimensions
joined_df = (Fact_CrimeStats
    .join(Dim_Municipality, "MunicipalityKey", "inner")
    .join(Dim_Violations, Fact_CrimeStats["CrimeTypeKey"] == Dim_Violations["Classification_Code"], "left").drop(Dim_Violations["Classification_Code"])
)

# Overview KPIs View
vw_Overview_KPIs = (joined_df
    .groupBy("YearKey", "MunicipalityName")
    .agg(
        _sum(col("ActualIncidents")).alias("ActualIncidents"),
        avg(col("CrimeRatePer100k")).alias("CrimeRatePer100k"),
        avg(col("ClearanceRate")).alias("ClearanceRate"),
        _sum(col("YouthCharges")).alias("YouthCharges"),
        _sum(col("AdultCharges")).alias("AdultCharges"),
        avg(col("PercentageChangeRate")).alias("PercentageChangeRate")
    )
)
vw_Overview_KPIs.write.format("delta").mode("overwrite").saveAsTable("vw_Overview_KPIs")

# Municipality Drilldown View
vw_Municipality_Drilldown = (joined_df
    .groupBy("YearKey", "MunicipalityName", "Member_Name")
    .agg(
        _sum(col("ActualIncidents")).alias("ActualIncidents"),
        avg(col("CrimeRatePer100k")).alias("CrimeRatePer100k"),
        avg(col("ClearanceRate")).alias("ClearanceRate"),
        _sum(col("YouthCharges")).alias("YouthCharges"),
        _sum(col("AdultCharges")).alias("AdultCharges")
    )
)
vw_Municipality_Drilldown.write.format("delta").mode("overwrite").saveAsTable("vw_Municipality_Drilldown")

# Comparative Analysis View
vw_Comparative_Analysis = (joined_df
    .groupBy("YearKey", "MunicipalityName")
    .agg(
        _sum(col("ActualIncidents")).alias("ActualIncidents"),
        avg(col("CrimeRatePer100k")).alias("CrimeRatePer100k"),
        avg(col("PercentageChangeRate")).alias("PercentageChangeRate")
    )
)
vw_Comparative_Analysis.write.format("delta").mode("overwrite").saveAsTable("vw_Comparative_Analysis")

# Display results
display(spark.table("vw_Overview_KPIs"))
display(spark.table("vw_Municipality_Drilldown"))
display(spark.table("vw_Comparative_Analysis"))




StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 91, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 5ce259b1-3a11-4e67-8b65-4ba6575fcccc)

SynapseWidget(Synapse.DataFrame, 1172d907-b308-4436-b6ab-0189931bde9d)

SynapseWidget(Synapse.DataFrame, c9f14686-b674-4865-9529-ca2ff88dcb2e)

In [9]:
# Count rows
# Count rows
df = spark.read.table("fact_crimestats")
row_count = df.filter(df["MunicipalityKey"] == "2021A000235").count()

print(f"Number of rows: {row_count}")

df2 = spark.read.table("tblcrime_data")
row_count2 = df2.filter(df2["DGUID"] == "2021A000235").count()
print(f"Number of row2s: {row_count2}")


StatementMeta(, 89be4b54-f490-47ec-887b-12f25f9ae53d, 11, Finished, Available, Finished)

Number of rows: 1400
Number of row2s: 0


In [1]:
sql_statement=f"ALTER TABLE tblsales_bronze RENAME TO tblcrime_bronze;";
spark.sql(sql_statement).show()

StatementMeta(, c6281c9f-d434-467f-9f38-2b2d35b89e59, 3, Finished, Available, Finished)

++
||
++
++



In [2]:
sql_statement=f"ALTER TABLE tblsales_silver RENAME TO tblcrime_silver;";
spark.sql(sql_statement).show()

StatementMeta(, c6281c9f-d434-467f-9f38-2b2d35b89e59, 4, Finished, Available, Finished)

++
||
++
++

